In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import os

# ------------------------------
# Paths
# ------------------------------
file_path = r"C:\Users\INFOTEC\OneDrive\Bureau\pre_w3w4\Cross_Week_Results\Week3_Final.xlsx"
output_folder = r"C:\Users\INFOTEC\Desktop\anomalie_w3\anomalie_statw3"
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, "Toutes_les_anomaliesw3.xlsx")

# ------------------------------
# Utils
# ------------------------------
def to_numeric_safe(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = (
                pd.to_numeric(
                    df[col].astype(str).str.replace(",", "."),
                    errors="coerce"
                )
                .fillna(0)
            )
    return df

# ------------------------------
# Read Excel
# ------------------------------
sheets = pd.read_excel(file_path, sheet_name=None)

# ------------------------------
# Writer (FORCÉ)
# ------------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for pays, df in sheets.items():
        print(f"▶ Traitement : {pays}")

        cols = ["Real Inventory (Qty)", "Stock Value (€)"]
        df = to_numeric_safe(df, cols)

        if df.empty:
            df.to_excel(writer, sheet_name=pays[:31], index=False)
            continue

        model = IsolationForest(
            n_estimators=200,
            contamination=0.01,
            random_state=42
        )

        model.fit(df[cols])
        df["anomaly"] = model.predict(df[cols])
        df["anomaly_score"] = model.decision_function(df[cols])

        anomalies = df[df["anomaly"] == -1].copy()

        # Même si 0 anomalies → sheet créé
        anomalies.to_excel(writer, sheet_name=pays[:31], index=False)

        print(f"✔ {len(anomalies)} anomalies")

print("\n✅ FICHIER CRÉÉ AVEC SUCCÈS :")
print(output_file)


▶ Traitement : Cyclam
✔ 10 anomalies
▶ Traitement : Germany
✔ 2 anomalies
▶ Traitement : India
✔ 5 anomalies
▶ Traitement : Korea
✔ 2 anomalies
▶ Traitement : Tianjin
✔ 1 anomalies
▶ Traitement : Kunshan
✔ 3 anomalies
▶ Traitement : SAME
✔ 1 anomalies
▶ Traitement : USA
✔ 5 anomalies
▶ Traitement : SCEET
✔ 5 anomalies

✅ FICHIER CRÉÉ AVEC SUCCÈS :
C:\Users\INFOTEC\Desktop\anomalie_w3\anomalie_statw3\Toutes_les_anomaliesw3.xlsx


In [ ]:
# Statistiques descriptives sur les colonnes numériques uniquement
print(anomalies.describe())


In [ ]:
# Installer matplotlib et seaborn directement depuis Jupyter
!pip install matplotlib seaborn --upgrade


In [ ]:
!pip install --user matplotlib seaborn


In [ ]:
# ==============================
# Histogramme global des scores d'anomalie
# ==============================
try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(10,6))
    sns.histplot(all_anomalies_df['anomaly_score'], bins=30, kde=True)
    plt.title("Distribution globale des scores d'anomalie (tous les pays)")
    plt.xlabel("Score d'anomalie")
    plt.ylabel("Nombre d'occurrences")
    plt.show()
except ImportError:
    print("Matplotlib ou Seaborn non installés, plots ignorés.")

# Top 10% anomalies les plus importantes (global)
top_n = max(1, int(len(all_anomalies_df) * 0.10))
top_anomalies = all_anomalies_df.nsmallest(top_n, 'anomaly_score')
print(f"\n=== Top 10% anomalies globales ===")
print(top_anomalies)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in df_numeric.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=anomalies[col])
    plt.title(f"Boxplot des anomalies pour {col}")
    plt.show()


In [ ]:
if len(df_numeric.columns) >= 2:
    cols = df_numeric.columns[:2]  # prendre les 2 premières colonnes numériques
    plt.figure(figsize=(6,4))
    plt.scatter(anomalies[cols[0]], anomalies[cols[1]], color='red', alpha=0.6)
    plt.xlabel(cols[0])
    plt.ylabel(cols[1])
    plt.title("Scatter plot des anomalies")
    plt.show()


In [ ]:
# ==============================
# Statistiques et top anomalies avec graphes affichés dans Python
# ==============================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# -----------------------------
# Chemins
# -----------------------------
input_file = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_w2\anomalie_stat_w2\Toutes_les_anomaliesw2.xlsx"
report_folder = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalieW2\reportsw2"
os.makedirs(report_folder, exist_ok=True)

# Fichier Excel de sortie pour stats & top anomalies
stats_file = os.path.join(report_folder, "anomalies_statistics_topw2.xlsx")

# -----------------------------
# Lire toutes les sheets
# -----------------------------
sheets = pd.read_excel(input_file, sheet_name=None)

# -----------------------------
# Concaténation globale
# -----------------------------
all_data = []
for sheet_name, df in sheets.items():
    df = df.copy()
    df['pays'] = sheet_name
    all_data.append(df)

all_data_df = pd.concat(all_data, ignore_index=True)

# -----------------------------
# Colonnes numériques
# -----------------------------
numeric_cols = all_data_df.select_dtypes(include=['float64', 'int64']).columns

# -----------------------------
# 1️⃣ Statistiques descriptives & top anomalies dans Excel
# -----------------------------
with pd.ExcelWriter(stats_file) as writer:
    # Statistiques descriptives
    for col in numeric_cols:
        if col in all_data_df.columns:
            all_data_df[col].describe().to_frame().to_excel(writer, sheet_name=f"Stats_{col[:25]}")
    
    # Top anomalies globales (ex: Stock Value élevé)
    if 'Stock Value (€)' in all_data_df.columns:
        top_percent = 0.10
        top_n = max(1, int(len(all_data_df) * top_percent))
        top_anomalies = all_data_df.sort_values('Stock Value (€)', ascending=False).head(top_n)
        top_anomalies.to_excel(writer, sheet_name="Top_Anomalies", index=False)

print(f"✅ Statistiques et top anomalies sauvegardées dans : {stats_file}")

# -----------------------------
# 2️⃣ Graphiques affichés directement dans Python
# -----------------------------
for col in numeric_cols:
    if col in all_data_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(all_data_df[col].dropna(), kde=True)
        plt.title(f"Histogramme : {col}")
        plt.show()

        plt.figure(figsize=(6,4))
        sns.boxplot(x=all_data_df[col].dropna())
        plt.title(f"Boxplot : {col}")
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Charger le fichier Excel
# -----------------------------
file = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w2w3\Cross_Week_Results\Week2_Final.xlsx"
sheets = pd.read_excel(file, sheet_name=None)
print("Feuilles trouvées :", list(sheets.keys()))

# -----------------------------
# Boucle sur chaque pays (chaque feuille)
# -----------------------------
for pays, df in sheets.items():
    print(f"\n▶ Traitement : {pays}")

    # Colonnes importantes
    target_col = "Weekly Target"
    usage_col = "Weekly Usage (Qty)"

    # Vérifier que les colonnes existent
    if target_col not in df.columns or usage_col not in df.columns:
        print(f"⚠️ Colonnes manquantes pour {pays}: {[c for c in [target_col, usage_col] if c not in df.columns]}")
        continue

    # Conversion en numérique
    df[target_col] = pd.to_numeric(df[target_col].astype(str).str.replace(',', '.').str.strip(), errors='coerce').fillna(0)
    df[usage_col] = pd.to_numeric(df[usage_col].astype(str).str.replace(',', '.').str.strip(), errors='coerce').fillna(0)

    # -----------------------------
    # Graphique avec deux courbes
    # -----------------------------
    plt.figure(figsize=(8,5))
    plt.plot(df.index, df[target_col], marker='o', linestyle='-', color='blue', label='Weekly Target')
    plt.plot(df.index, df[usage_col], marker='o', linestyle='--', color='orange', label='Weekly Usage')

    plt.title(f"Weekly Target vs Weekly Usage – {pays}")
    plt.xlabel("Index")
    plt.ylabel("Quantity")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# =================================================
# FICHIERS - MODIFIE CES CHEMINS SELON TON ORDINATEUR
# =================================================
input_file = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w2w3\Cross_Week_Results\Week2_Final.xlsx"
output_dir = r"C:\Users\INFOTEC\OneDrive\Bureau\anomalie_W2\anomalie_metier\\"

# Créer les répertoires de sortie s'ils n'existent pas
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir + 'pays_files\\', exist_ok=True)

output_file_all = output_dir + 'Week2_With_Anomalies_Metier.xlsx'
output_file_anomaly = output_dir + 'Anomalies_Metier_Onlyw2.xlsx'

# =================================================
# FONCTIONS DE DÉTECTION D'ANOMALIES
# =================================================
def convert_to_numeric(value):
    """Convertir une valeur en numérique (gérer virgules, espaces, etc.)"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    # Convertir string
    try:
        str_val = str(value).replace(',', '.').replace(' ', '').strip()
        return float(str_val)
    except:
        return np.nan

def detect_inventory_negatif(row):
    """Détecte si inventory est négatif"""
    inv = convert_to_numeric(row.get("Real Inventory (Qty)", None))
    if pd.isna(inv):
        return 0
    return 1 if inv < 0 else 0

def detect_rupture_stock(row):
    """Détecte rupture de stock (inventory = 0)"""
    inv = convert_to_numeric(row.get("Real Inventory (Qty)", None))
    if pd.isna(inv):
        return 0
    return 1 if inv == 0 else 0

def detect_rotation_rapide(row):
    """Détecte rotation trop rapide (WU < 0.5)"""
    wu = convert_to_numeric(row.get("WU", None))
    if pd.isna(wu):
        return 0
    return 1 if wu < 0.5 else 0

def detect_sous_stock(row):
    """Détecte sous-stock (Inventory < Target)"""
    inv = convert_to_numeric(row.get("Real Inventory (Qty)", None))
    target = convert_to_numeric(row.get("Weekly Target", None))
    if pd.isna(inv) or pd.isna(target):
        return 0
    return 1 if (inv < target and inv > 0) else 0

def detect_surstock_critique(row):
    """Détecte sur-stock critique (WU > 6)"""
    wu = convert_to_numeric(row.get("WU", None))
    if pd.isna(wu):
        return 0
    return 1 if wu > 6 else 0

def detect_metier_principal(row):
    """Détecte l'anomalie principale (la plus critique)"""
    inv = convert_to_numeric(row.get("Real Inventory (Qty)", None))
    wu = convert_to_numeric(row.get("WU", None))
    target = convert_to_numeric(row.get("Weekly Target", None))

    if pd.isna(inv) or pd.isna(wu) or pd.isna(target):
        return "Normal"

    # Ordre de priorité des anomalies
    if inv < 0:
        return "Inventory négatif"
    if inv == 0:
        return "Rupture de stock"
    if wu < 0.5:
        return "Rotation trop rapide"
    if inv < target:
        return "Sous-stock"
    if wu > 6:
        return "Sur-stock critique"

    return "Normal"

# =================================================
# TRAITEMENT PAR SHEET (PAR PAYS)
# =================================================
print("📂 Chargement du fichier Excel...")
xl_file = pd.ExcelFile(input_file)
sheet_names = xl_file.sheet_names

print(f"✅ {len(sheet_names)} sheets trouvés: {sheet_names}")

# Dictionnaires pour stocker les résultats
all_data_with_anomalies = {}
only_anomalies = {}

# Statistiques globales
global_stats = {
    'total_lignes': 0,
    'total_anomalies': 0,
    'inventory_negatif': 0,
    'rupture_stock': 0,
    'rotation_rapide': 0,
    'sous_stock': 0,
    'surstock_critique': 0
}

stats_by_country = []

print("\n🔍 Détection des anomalies par pays...")
print("=" * 70)

# Traiter chaque sheet (pays)
for sheet_name in sheet_names:
    print(f"\n📍 Traitement de: {sheet_name}")
    
    # Charger les données du sheet
    df = pd.read_excel(input_file, sheet_name=sheet_name)
    print(f"   Lignes: {len(df)}")
    
    # Appliquer les détections d'anomalies
    df['Anomalie_Metier'] = df.apply(detect_metier_principal, axis=1)
    df['Anomalie_Inventory_Negatif'] = df.apply(detect_inventory_negatif, axis=1)
    df['Anomalie_Rupture_Stock'] = df.apply(detect_rupture_stock, axis=1)
    df['Anomalie_Rotation_Rapide'] = df.apply(detect_rotation_rapide, axis=1)
    df['Anomalie_Sous_Stock'] = df.apply(detect_sous_stock, axis=1)
    df['Anomalie_Surstock_Critique'] = df.apply(detect_surstock_critique, axis=1)
    
    # Colonne total anomalies
    df['Total_Anomalies'] = (
        df['Anomalie_Inventory_Negatif'] + 
        df['Anomalie_Rupture_Stock'] + 
        df['Anomalie_Rotation_Rapide'] + 
        df['Anomalie_Sous_Stock'] + 
        df['Anomalie_Surstock_Critique']
    )
    
    # Stocker toutes les données
    all_data_with_anomalies[sheet_name] = df
    
    # Filtrer seulement les anomalies
    df_anomalies = df[df['Total_Anomalies'] > 0].copy()
    only_anomalies[sheet_name] = df_anomalies
    
    # Statistiques pour ce pays
    nb_anomalies = len(df_anomalies)
    pct_anomalies = (nb_anomalies / len(df) * 100) if len(df) > 0 else 0
    
    stats_by_country.append({
        'Pays': sheet_name,
        'Total_Lignes': len(df),
        'Inventory_Negatif': int(df['Anomalie_Inventory_Negatif'].sum()),
        'Rupture_Stock': int(df['Anomalie_Rupture_Stock'].sum()),
        'Rotation_Rapide': int(df['Anomalie_Rotation_Rapide'].sum()),
        'Sous_Stock': int(df['Anomalie_Sous_Stock'].sum()),
        'Surstock_Critique': int(df['Anomalie_Surstock_Critique'].sum()),
        'Total_Anomalies': int(df['Total_Anomalies'].sum()),
        'Lignes_avec_Anomalie': nb_anomalies,
        'Pourcentage_Anomalies': round(pct_anomalies, 2)
    })
    
    # Mettre à jour statistiques globales
    global_stats['total_lignes'] += len(df)
    global_stats['total_anomalies'] += nb_anomalies
    global_stats['inventory_negatif'] += int(df['Anomalie_Inventory_Negatif'].sum())
    global_stats['rupture_stock'] += int(df['Anomalie_Rupture_Stock'].sum())
    global_stats['rotation_rapide'] += int(df['Anomalie_Rotation_Rapide'].sum())
    global_stats['sous_stock'] += int(df['Anomalie_Sous_Stock'].sum())
    global_stats['surstock_critique'] += int(df['Anomalie_Surstock_Critique'].sum())
    
    print(f"   ✅ Anomalies: {nb_anomalies} ({pct_anomalies:.2f}%)")
    
    # =========================================
    # CRÉER UN FICHIER EXCEL PAR PAYS
    # =========================================
    country_file = output_dir + f'pays_files\\{sheet_name}_Anomalies.xlsx'
    df.to_excel(country_file, index=False, sheet_name=sheet_name)
    print(f"   💾 Fichier créé: {sheet_name}_Anomalies.xlsx")

print("\n" + "=" * 70)

# =================================================
# STATISTIQUES GLOBALES
# =================================================
print("\n📊 STATISTIQUES GLOBALES:")
print("=" * 70)
print(f"Total lignes: {global_stats['total_lignes']}")
print(f"Lignes avec anomalies: {global_stats['total_anomalies']}")
print(f"Pourcentage anomalies: {(global_stats['total_anomalies'] / global_stats['total_lignes'] * 100):.2f}%")
print()
print("Détail par type:")
print(f"  • Inventory négatif: {global_stats['inventory_negatif']}")
print(f"  • Rupture de stock: {global_stats['rupture_stock']}")
print(f"  • Rotation rapide: {global_stats['rotation_rapide']}")
print(f"  • Sous-stock: {global_stats['sous_stock']}")
print(f"  • Sur-stock critique: {global_stats['surstock_critique']}")
print("=" * 70)

# =================================================
# TABLEAU RÉCAPITULATIF PAR PAYS
# =================================================
print("\n📋 RÉCAPITULATIF PAR PAYS:")
df_stats = pd.DataFrame(stats_by_country)
print(df_stats.to_string(index=False))

# =================================================
# FICHIER 1: TOUTES LES DONNÉES AVEC ANOMALIES
# =================================================
print(f"\n💾 Création du fichier complet avec anomalies...")
with pd.ExcelWriter(output_file_all, engine='openpyxl') as writer:
    # Sheet résumé
    df_stats.to_excel(writer, sheet_name='RÉSUMÉ', index=False)
    
    # Chaque pays dans un sheet
    for sheet_name, df_data in all_data_with_anomalies.items():
        df_data.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Fichier créé: Week1_With_Anomalies_Metier.xlsx")
print(f"   📊 {len(all_data_with_anomalies)} sheets (1 par pays)")

# =================================================
# FICHIER 2: SEULEMENT LES ANOMALIES
# =================================================
print(f"\n💾 Création du fichier anomalies seulement...")
with pd.ExcelWriter(output_file_anomaly, engine='openpyxl') as writer:
    # Sheet résumé
    df_stats.to_excel(writer, sheet_name='RÉSUMÉ', index=False)
    
    # Chaque pays dans un sheet (seulement anomalies)
    for sheet_name, df_data in only_anomalies.items():
        if len(df_data) > 0:
            df_data.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Fichier créé: Anomalies_Metier_Only.xlsx")
print(f"   ⚠️  Seulement les lignes avec anomalies")

# =================================================
# RÉSUMÉ FINAL
# =================================================
print("\n" + "=" * 70)
print("✨ TRAITEMENT TERMINÉ!")
print("=" * 70)
print(f"\n📁 FICHIERS CRÉÉS:")
print(f"\n1️⃣  Week1_With_Anomalies_Metier.xlsx")
print(f"    • Toutes les données + colonnes anomalies")
print(f"    • {len(all_data_with_anomalies)} sheets (1 par pays)")
print(f"\n2️⃣  Anomalies_Metier_Only.xlsx")
print(f"    • Seulement les lignes avec anomalies")
print(f"    • {len(only_anomalies)} sheets (1 par pays)")
print(f"\n3️⃣  Dossier 'pays_files/' avec {len(sheet_names)} fichiers Excel:")
for sheet_name in sheet_names:
    nb_lignes = len(all_data_with_anomalies[sheet_name])
    nb_anomalies = len(only_anomalies[sheet_name])
    print(f"    • {sheet_name}_Anomalies.xlsx ({nb_lignes} lignes, {nb_anomalies} anomalies)")
print(f"\n📋 COLONNES AJOUTÉES À CHAQUE FICHIER:")
print(f"  • Anomalie_Metier: Type principal (ou 'Normal')")
print(f"  • Anomalie_Inventory_Negatif: 1 = oui, 0 = non")
print(f"  • Anomalie_Rupture_Stock: 1 = oui, 0 = non")
print(f"  • Anomalie_Rotation_Rapide: 1 = oui, 0 = non")
print(f"  • Anomalie_Sous_Stock: 1 = oui, 0 = non")
print(f"  • Anomalie_Surstock_Critique: 1 = oui, 0 = non")
print(f"  • Total_Anomalies: Somme (0 si normal)")
print("=" * 70) 

# Pause pour voir les résultats
input("\nAppuie sur Entrée pour fermer...")